In [1]:
import html as html_lib
import re
from html import unescape
from itertools import combinations_with_replacement

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

print("Imports OK — run the next cell (helpers), then the UI cell.")

Imports OK — run the next cell (helpers), then the UI cell.


In [2]:
BASE = "https://fantasybumps.org.uk"


def interpolated_costs(n_crews: int) -> np.ndarray:
    """When the event has concluded and buy buttons are gone: head 300, foot 20, linear by row order."""
    if n_crews <= 0:
        return np.array([], dtype=int)
    if n_crews == 1:
        return np.array([300], dtype=int)
    r = np.arange(n_crews, dtype=float)
    return np.round(300 - 280 * r / (n_crews - 1)).astype(int)


def parse_signed_int(val: str | None, default: int = 0) -> int:
    if val is None:
        return default
    s = str(val).strip().replace("\u2212", "-")
    if s.startswith("+"):
        s = s[1:]
    try:
        return int(s)
    except ValueError:
        return default


def parse_market_page(html: str) -> tuple[pd.DataFrame, bool]:
    """Returns (df, costs_were_interpolated). Interpolation runs only if the page says the event concluded."""
    soup = BeautifulSoup(html, "html.parser")
    concluded = "this event has concluded" in html.lower()
    rows = []
    for btn in soup.select("button.payout"):
        row_el = btn.find_parent("div", class_="market-row")
        if row_el is None:
            continue
        name_el = row_el.find("div", class_="flex-grow-1")
        if name_el is None:
            continue
        name = unescape(name_el.get_text(strip=True))
        bump_up = parse_signed_int(btn.get("data-bump-up"))
        row_over = parse_signed_int(btn.get("data-row-over"))
        bumped_dn = parse_signed_int(btn.get("data-bumped-down"))
        pop = btn.get("data-popularity")
        try:
            popularity = float(pop) if pop is not None else np.nan
        except ValueError:
            popularity = np.nan
        buy = row_el.find("button", class_=lambda c: c and "btn-buy" in c)
        price: int | None = None
        if buy is not None:
            for attr in ("data-cost", "data-price"):
                v = buy.get(attr)
                if v is not None and str(v).strip() != "":
                    price = parse_signed_int(v)
                    break
            if price is None:
                t = buy.get_text()
                m = re.search(r"(\d+)", t)
                if m:
                    price = int(m.group(1))
        rows.append(
            {
                "crew": name,
                "return": bump_up,
                "row_over": row_over,
                "bumped_down": bumped_dn,
                "popularity": popularity,
                "cost": price,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        return df, False
    df["cost"] = df["cost"].astype("Int64")
    if concluded:
        df["cost"] = interpolated_costs(len(df))
        return df, True
    return df, False


def fetch_event(series: str, year: int, gender: str) -> tuple[pd.DataFrame, bool]:
    """series: 'eights' | 'torpids'; gender: 'men' | 'women'. Returns (df, used_interpolation)."""
    url = f"{BASE}/{series}{year}/{gender}/"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return parse_market_page(r.text)


def latest_year_for_series(series: str, min_y: int = 2020, max_y: int = 2030) -> int | None:
    for y in range(max_y, min_y - 1, -1):
        u = f"{BASE}/{series}{y}/men/"
        try:
            h = requests.head(u, timeout=10, allow_redirects=True)
            if h.status_code == 200:
                return y
            g = requests.get(u, timeout=10, allow_redirects=True)
            if g.status_code == 200 and "Fantasy Bumps" in g.text:
                return y
        except requests.RequestException:
            continue
    return None


def run_optimization(crews: pd.Series, costs: np.ndarray, returns: np.ndarray, budget: int, top_n: int = 10):
    """Same logic as the original notebook: multiset size 9, maximize return under budget."""
    n = len(crews)
    if n == 0:
        return []
    all_combinations = np.array(list(combinations_with_replacement(range(n), 9)))
    combination_costs = costs[all_combinations].sum(axis=1)
    combination_returns = returns[all_combinations].sum(axis=1)
    valid_mask = combination_costs <= budget
    valid_combinations = all_combinations[valid_mask]
    valid_returns = combination_returns[valid_mask]
    valid_costs = combination_costs[valid_mask]
    if len(valid_returns) == 0:
        return []
    top_indices = np.argsort(valid_returns)[-top_n:][::-1]
    out = []
    for rank, top_idx in enumerate(top_indices):
        comb = valid_combinations[top_idx]
        best_crews = crews.iloc[comb].values
        tot_c = valid_costs[top_idx]
        tot_r = valid_returns[top_idx]
        detail = pd.DataFrame({"crew": best_crews, "cost": costs[comb], "return": returns[comb]})
        out.append({"rank": rank + 1, "total_cost": tot_c, "total_return": tot_r, "detail": detail})
    return out


def format_results(title: str, results: list) -> str:
    if not results:
        return f"<p><b>{html_lib.escape(title)}</b>: no valid combinations (check budget or picks).</p>"
    parts = [f"<h4>{html_lib.escape(title)}</h4>"]
    for item in results:
        parts.append(
            f"<p><b>Combination {item['rank']}</b> &mdash; cost {item['total_cost']}, return {item['total_return']}</p>"
        )
        parts.append(item["detail"].to_html(index=False))
    return "\n".join(parts)


In [ ]:
state = {"men": None, "women": None}

series_dd = widgets.Dropdown(
    options=[("Summer Eights", "eights"), ("Torpids", "torpids")],
    value="eights",
    description="Series:",
)
year_text = widgets.BoundedIntText(value=2025, min=2020, max=2035, description="Year:")
use_latest = widgets.Checkbox(value=False, description="Use latest year (probe /men/)")
budget_m = widgets.BoundedIntText(value=1000, min=0, max=5000, description="Crabs men:")
budget_w = widgets.BoundedIntText(value=1000, min=0, max=5000, description="Crabs women:")
load_btn = widgets.Button(description="Load crews from site", button_style="primary")
opt_btn = widgets.Button(description="Run optimization", button_style="success")
status = widgets.HTML("")
out_area = widgets.Output()

men_box = widgets.VBox(
    [],
    layout=widgets.Layout(max_height="280px", overflow_y="auto", border="1px solid #ccc", padding="6px"),
)
women_box = widgets.VBox(
    [],
    layout=widgets.Layout(max_height="280px", overflow_y="auto", border="1px solid #ccc", padding="6px"),
)

men_checks: list[widgets.Checkbox] = []
women_checks: list[widgets.Checkbox] = []


def rebuild_checks(df: pd.DataFrame | None) -> tuple[list[widgets.Checkbox], widgets.VBox]:
    cbs: list[widgets.Checkbox] = []
    if df is None or df.empty:
        return cbs, widgets.VBox([widgets.Label("Load crews first.")])
    priced = df.loc[df["cost"].notna()].copy()
    if priced.empty:
        return cbs, widgets.VBox([widgets.Label("No crews with a usable cost.")])
    for _, r in priced.iterrows():
        name = str(r["crew"])
        tip = f"{name} — +1 payout {int(r['return'])}, cost {int(r['cost'])}"
        if "popularity" in df.columns and pd.notna(r.get("popularity")):
            tip += f", pop {float(r['popularity']):.2f}"
        desc = name if len(name) <= 64 else name[:61] + "..."
        cb = widgets.Checkbox(
            value=False,
            description=desc,
            tooltip=tip,
            indent=False,
            layout=widgets.Layout(width="98%"),
        )
        cbs.append(cb)
    return cbs, widgets.VBox(cbs)


def on_load(_=None):
    status.value = "Loading…"
    ser = series_dd.value
    if use_latest.value:
        y = latest_year_for_series(ser)
        if y is None:
            status.value = "<span style='color:red'>Could not detect latest year.</span>"
            return
        year_text.value = y
    y = int(year_text.value)
    try:
        men_df, men_interp = fetch_event(ser, y, "men")
        women_df, women_interp = fetch_event(ser, y, "women")
    except Exception as e:
        status.value = f"<span style='color:red'>Fetch failed: {html_lib.escape(str(e))}</span>"
        return
    state["men"] = (
        men_df if men_interp else men_df.dropna(subset=["cost"]).reset_index(drop=True)
    )
    state["women"] = (
        women_df if women_interp else women_df.dropna(subset=["cost"]).reset_index(drop=True)
    )
    global men_checks, women_checks
    men_checks, mb = rebuild_checks(state["men"])
    women_checks, wb = rebuild_checks(state["women"])
    men_box.children = (mb,)
    women_box.children = (wb,)
    parts = [f"Loaded <b>{ser}{y}</b>. "]
    if men_interp:
        parts.append(
            f"Men: <b>{len(men_df)}</b> crews — costs <b>estimated</b> (event concluded; 300→20 by order on page). "
        )
    else:
        nm = int(men_df["cost"].isna().sum()) if len(men_df) else 0
        npm = int(men_df["cost"].notna().sum())
        parts.append(f"Men: <b>{npm}</b> with buy price from site")
        if nm:
            parts.append(f" ({nm} skipped, no price on page)")
        parts.append(". ")
    if women_interp:
        parts.append(
            f"Women: <b>{len(women_df)}</b> crews — costs <b>estimated</b> (event concluded; 300→20). "
        )
    else:
        nw = int(women_df["cost"].isna().sum()) if len(women_df) else 0
        npw = int(women_df["cost"].notna().sum())
        parts.append(f"Women: <b>{npw}</b> with buy price from site")
        if nw:
            parts.append(f" ({nw} skipped, no price on page)")
        parts.append(". ")
    if not men_interp and not women_interp:
        if (len(men_df) and men_df["cost"].notna().sum() == 0) and (
            len(women_df) and women_df["cost"].notna().sum() == 0
        ):
            parts.append(
                "<span style='color:red'><b>No prices — cannot optimize.</b></span>"
            )
    status.value = "".join(parts)


def on_opt(_=None):
    with out_area:
        clear_output()
        print("HAVE YOU UPDATED YOUR CRAB BALANCES?")
        if state["men"] is None or state["women"] is None:
            print("Load crews first.")
            return
        mx = sum(cb.value for cb in men_checks)
        wx = sum(cb.value for cb in women_checks)
        if mx < 1 and wx < 1:
            print("Select at least one predicted +1 crew for men and/or women.")
            return
        bm, bw = int(budget_m.value), int(budget_w.value)
        if mx >= 1:
            mdf = state["men"].iloc[[i for i, cb in enumerate(men_checks) if cb.value]].reset_index(drop=True)
            m_res = run_optimization(mdf["crew"], mdf["cost"].values, mdf["return"].values, bm)
            display(HTML(format_results("Men's boat (top combinations)", m_res)))
        if wx >= 1:
            wdf = state["women"].iloc[[i for i, cb in enumerate(women_checks) if cb.value]].reset_index(drop=True)
            w_res = run_optimization(wdf["crew"], wdf["cost"].values, wdf["return"].values, bw)
            display(HTML(format_results("Women's boat (top combinations)", w_res)))


load_btn.on_click(on_load)
opt_btn.on_click(on_opt)

ui = widgets.VBox(
    [
        widgets.HTML("<h3>Fantasy Bumps — interactive +1 optimizer</h3>"),
        widgets.HBox([series_dd, year_text, use_latest]),
        widgets.HBox([budget_m, budget_w]),
        load_btn,
        status,
        widgets.HBox(
            [
                widgets.VBox([widgets.Label("Predicted +1 (men) — tick crews:"), men_box]),
                widgets.VBox([widgets.Label("Predicted +1 (women) — tick crews:"), women_box]),
            ],
            layout=widgets.Layout(gap="12px", width="100%"),
        ),
        opt_btn,
        out_area,
    ],
    layout=widgets.Layout(width="100%", max_width="1280px"),
)

import sys

print("=" * 64)
print("Below you should see dropdowns and buttons. If this text is ALL you see,")
print("ipywidgets is not rendering (common in some notebook UIs). Try:")
print("  • Cursor: Jupyter extension enabled; pick kernel Python (fantasy_bumps)")
print("  • Trust the notebook if prompted")
print("  • Or run in a browser:")
print('    conda activate fantasy_bumps && jupyter notebook fantasy_bumps.ipynb')
print("Python:", sys.executable)
print("ipywidgets:", widgets.__version__)
print("=" * 64)
display(HTML("<p><b>Interactive controls:</b></p>"))
display(ui)


Below you should see dropdowns and buttons. If this text is ALL you see,
ipywidgets is not rendering (common in some notebook UIs). Try:
  • Cursor: Jupyter extension enabled; pick kernel Python (fantasy_bumps)
  • Trust the notebook if prompted
  • Or run in a browser:
    conda activate fantasy_bumps && jupyter notebook fantasy_bumps.ipynb
Python: /usr/local/Caskroom/miniconda/base/envs/fantasy_bumps/bin/python
ipywidgets: 8.1.8


In [4]:
# Optional: original CSV workflow (same optimizer as run_optimization)
# file_path = './Fantasy bumps - Sheet1.csv'
# data = pd.read_csv(file_path).dropna(subset=["crew"])
# crews = data["crew"]
# costs = data["cost"].values.astype(int)
# returns = data["return"].values.astype(int)
# for item in run_optimization(crews, costs, returns, budget=2199):
#     print(item["detail"])
#     print(item["total_cost"], item["total_return"])